In [1]:
import numpy as np
from pathlib import Path

In [2]:
LONG_COMP = ['VCVS', 'CCVS', 'VCCS', 'CCCS', 'ISC']
BASIC = ['R', 'V', 'I', 'C', 'L']
NLIN = ['D']

def get_type(name):
    name = name.upper()
    for prefix in LONG_COMP:
        if name.startswith(prefix):
            return prefix
    return name[0]

def val_comp_format(name, comp_type, comp):
    #Supported component validation
    if comp_type not in BASIC and comp_type not in LONG_COMP and comp_type not in NLIN:
        raise ValueError('Unsupported component type', name)

    #Component format validation
    if comp_type == 'ISC':
        if len(comp) != 3:
            raise ValueError('Expected ISCname node+ node-', name)
        if comp[1] == comp[2]:
            raise ValueError('Sensing-short terminals must differ', name)
    elif comp_type in ('V', 'I'):
        if len(comp) != 5 or comp[3] != 'DC':
            raise ValueError('Expected name node+ node- DC value', name)
    elif comp_type in ('R', 'C', 'L'):
        if len(comp) != 4:
            raise ValueError('Expected name node+ node- value', name)
    elif comp_type in ('VCVS', 'VCCS'):
        if len(comp) != 6:
            raise ValueError('Expected name out+ out- control+ control- gain', name)
    elif comp_type in ('CCCS', 'CCVS'):
        if len(comp) != 5:
            raise ValueError('Expected name out+ out- controlling_component gain', name)
    elif comp_type == 'D':
        if len(comp) != 3:
            raise ValueError('Expected name anode cathode', name)
                         
def read_netlist(file):
    comps, dirs = [], []
    seen_names = set()
    found_end = False
    with open(file) as f:
        title = f.readline().rstrip('\n') 
        for line in f:
            line = line.strip().upper()

            #empty space or comments
            if not line or line.startswith('*'):
                continue

            #.end
            if found_end:
                raise ValueError('Content after .END', line)
            if line == '.END':
                found_end = True
                continue

            #directive control
            if line.startswith('.'):
                dirs.append(line)
                continue
                
            parts = line.split()
            name = parts[0]
            comp_type = get_type(name)
            val_comp_format(name, comp_type, parts)


            if name in seen_names:
                raise ValueError('Duplicate component', name)
            seen_names.add(name)
            comps.append(parts)

    if not found_end:
        raise ValueError('Missing .END')
    if len(dirs) != 1:
        raise ValueError('Expected exactly one active directive')
    if not comps:
        raise ValueError('Cannot calculate circuit with no components')
    return title, comps, dirs[0]

In [3]:
def gen_matrices(comps):
    node_index, cur_index = {}, {}
    size = 0
    for comp in comps:
        name = comp[0]
        kind = get_type(name)
        nodes = comp[1:3]
        if kind in ('VCVS', 'VCCS'):
            nodes = nodes + comp[3:5]
        for node in nodes:
            if node not in node_index:
                if node != '0':
                    node_index[node] = size
                    size += 1 
                else:
                    node_index[node] = None
        if kind in ('V', 'VCVS', 'CCVS', 'L', 'ISC'):
            cur_index['I' + name] = size
            size += 1
    
    G = np.zeros((size, size), dtype=float)
    C = np.zeros((size, size), dtype=float)
    b = np.zeros(size, dtype=float)
    return G, C, b, node_index, cur_index

In [4]:
def numeric_value(text, name):
    try:
        return float(text)
    except ValueError:
        raise ValueError(f'{name}: expected a numerical value, got {text!r}') from None

def fill_matrices(comps, G, C, b, node_index, cur_index):
    def add_G(row, col, value):
        if row is not None and col is not None:
            G[row, col] += value

    def add_C(row, col, value):
        if row is not None and col is not None:
            C[row, col] += value

    def add_b(row, value):
        if row is not None:
            b[row] += value
            
    def stamp_voltage_branch(i, j, current):
        add_G(j, current, -1)
        add_G(current, j, -1)
        add_G(i, current, 1)
        add_G(current, i, 1)

    G.fill(0.0)
    C.fill(0.0)
    b.fill(0.0)
    diodes = []
    for comp in comps:
        name = comp[0]
        comp_type = get_type(name)

        i = node_index[comp[1]]
        j = node_index[comp[2]]

        if comp_type == 'R':
            r = numeric_value(comp[3], name)

            if not np.isfinite(r) or r <= 0:
                raise ValueError('Resistance must be finite and positive')

            g = 1.0 / r

            add_G(i, i, g)
            add_G(i, j, -g)
            add_G(j, i, -g)
            add_G(j, j, g)

        elif comp_type == 'I':
            current = numeric_value(comp[4], name)

            if not np.isfinite(current):
                raise ValueError('Current source must be finite')

            add_b(i, -current)
            add_b(j, current)

        elif comp_type == 'V':
            voltage = numeric_value(comp[4], name)
            ie = cur_index['I' + name]

            if not np.isfinite(voltage):
                raise ValueError('Voltage source must be finite')

            stamp_voltage_branch(i, j, ie)
            add_b(ie, voltage)

        elif comp_type == 'ISC':
            ie = cur_index['I' + name]
            stamp_voltage_branch(i, j, ie)

        elif comp_type == 'C':
            F = numeric_value(comp[3], name)

            if not np.isfinite(F) or F < 0:
                raise ValueError('Capacitance must be finite and non-negative')

            add_C(i, i, F)
            add_C(i, j, -F)
            add_C(j, i, -F)
            add_C(j, j, F)

        elif comp_type == 'L':
            H = numeric_value(comp[3], name)

            if not np.isfinite(H) or H < 0:
                raise ValueError('Inductance must be finite and non-negative')

            il = cur_index['I' + name]

            stamp_voltage_branch(i, j, il)
            add_C(il, il, -H)

        elif comp_type == 'VCCS':
            cplus = node_index[comp[3]]
            cminus = node_index[comp[4]]
            gm = numeric_value(comp[5], name)

            if not np.isfinite(gm):
                raise ValueError('VCCS gain must be finite')

            add_G(i, cplus, gm)
            add_G(i, cminus, -gm)
            add_G(j, cplus, -gm)
            add_G(j, cminus, gm)

        elif comp_type == 'VCVS':
            cplus = node_index[comp[3]]
            cminus = node_index[comp[4]]
            A = numeric_value(comp[5], name)
            ie = cur_index['I' + name]

            if not np.isfinite(A):
                raise ValueError('VCVS gain must be finite')

            stamp_voltage_branch(i, j, ie)
            add_G(ie, cplus, -A)
            add_G(ie, cminus, A)

        elif comp_type == 'CCCS':
            isense = cur_index['I' + comp[3]]
            gain = numeric_value(comp[4], name)

            if not np.isfinite(gain):
                raise ValueError('CCCS gain must be finite')

            add_G(i, isense, gain)
            add_G(j, isense, -gain)

        elif comp_type == 'CCVS':
            ie = cur_index['I' + name]
            isense = cur_index['I' + comp[3]]
            gain = numeric_value(comp[4], name)

            if not np.isfinite(gain):
                raise ValueError('CCVS gain must be finite')

            stamp_voltage_branch(i, j, ie)
            add_G(ie, isense, -gain)

        elif comp_type == 'D':
            diodes.append(comp)
            
    return G, C, b, diodes
    
def fill_diodes(diodes, x, node_index, Is=1e-15, Vt=0.025):
    size = len(x)
    f = np.zeros(size, dtype=float)
    J = np.zeros((size, size), dtype=float)

    with np.errstate(over='raise', invalid='raise', divide='raise'):
        for diode in diodes:
            i = node_index[diode[1]]
            j = node_index[diode[2]]

            vi = 0.0 if i is None else x[i]
            vj = 0.0 if j is None else x[j]
            vd = vi - vj

            exp_term = np.exp(vd / Vt)
            Id = Is * np.expm1(vd / Vt)
            gd = (Is / Vt) * exp_term

            if i is not None:
                f[i] += Id
                J[i, i] += gd

            if j is not None:
                f[j] -= Id
                J[j, j] += gd

            if i is not None and j is not None:
                J[i, j] -= gd
                J[j, i] -= gd

    return f, J

In [5]:
def lin_dc_analysis(G, b):
    try: 
        x = np.linalg.solve(G, b)
    except np.linalg.LinAlgError as e:
        raise ValueError('No unique DC Solution') from e
    return x

def nlin_dc_analysis(G, b, diodes, node_index, estep_max=1.0, cont_steps=50):
    x0 = np.zeros(len(b), dtype=float)
    try:
        x, estep = newton_raphson(G, b, x0, diodes, node_index, alpha=1.0)
        if estep <= estep_max:
            return x
    
    #if it runs out of iterations or runs into error, try with continuation
        print('Relative-step check failed; trying continuation.')
    except (RuntimeError, FloatingPointError,np.linalg.LinAlgError) as e:
        print(f'Normal Newton-Raphson failed: {e}')
        print('Trying continuation.')

    if not isinstance(cont_steps, int) or cont_steps <= 0:
        raise ValueError('cont_steps must be a positive integer')

    #Newton-Raphson with Continuation
    x = np.zeros(len(b), dtype=float)
    for step in range(1, cont_steps + 1):
        alpha = step / cont_steps

        try:
            x, estep = newton_raphson(G, b, x, diodes, node_index, alpha)
        except (RuntimeError,FloatingPointError,np.linalg.LinAlgError) as e:
            raise RuntimeError(f'Continuation failed at alpha={alpha}') from e

    return x

def newton_raphson(G, b, x, diodes, node_index, alpha=1.0, max_iters=100, thres=1e-6, denom_floor=1e-12):
    x = np.array(x, dtype=float, copy=True)
    estep = 100
    with np.errstate(over='raise', invalid='raise', divide='raise'):
        for iteration in range(max_iters):
            f, Jnl = fill_diodes(diodes, x, node_index)

            F = G @ x + f - alpha * b
            J = G + Jnl

            delta_x = np.linalg.solve(J, -F)
            x = x + delta_x

            if not np.all(np.isfinite(x)):
                raise FloatingPointError('Nonfinite Newton iterate')

            if np.linalg.norm(delta_x) <= thres:
                estep = (np.linalg.norm(delta_x)/ max(np.linalg.norm(x), denom_floor))
                return x, estep

    raise RuntimeError(f'Newton-Raphson did not converge within {max_iters} iterations')

In [6]:
def run_netlist(file='data.txt', show_matrices=True):
    title, comps, act_dir = read_netlist(file)
    G, C, b, node_index, cur_index = gen_matrices(comps)

    #catching numpy errors
    with np.errstate(over='raise', invalid='raise', divide='raise'):
        try:
            G, C, b, diodes = fill_matrices(comps, G, C, b, node_index, cur_index)
        except FloatingPointError as e:
            raise ValueError('Numerical overflow while building matrices') from e
    if show_matrices:
        print(node_index)
        print(cur_index)
        print(G)
        print(b)

    #directive handling
    if act_dir == '.DC':
        if diodes:
            print('Running nonlinear DC analysis')
            x = nlin_dc_analysis(G, b, diodes, node_index)
        else:
            print('Running linear DC analysis')
            x = lin_dc_analysis(G, b)

    for key in node_index:
        if key == '0':
            continue
        index = node_index[key]
        val = x[index]
        print(key + ":" + str(val) + ' V')
    for key in cur_index:
        index = cur_index[key]
        val = x[index]
        print(key + ":" + str(val) + ' A')
    return G, C, b, node_index, cur_index, x

G, C, b, node_index, cur_index, x = run_netlist()

{'N1': 0, '0': None, 'N2': 2}
{'IVIN': 1}
[[0.   1.   0.  ]
 [1.   0.   0.  ]
 [0.   0.   0.02]]
[0. 1. 0.]
Running nonlinear DC analysis
N1:1.0 V
N2:0.2673144090476317 V
IVIN:-0.005346288180952634 A


In [7]:
from pathlib import Path
import contextlib
import io
import math
import shutil

from compare_spice import run_ngspice

NGSPICE = "C:\\Users\\patri\\Downloads\\ngspice-47_64\\Spice64\\bin\\ngspice.exe"

# Otherwise, uncomment and adjust this Windows example:
# NGSPICE = r"C:\Spice64\bin\ngspice.exe"

if not NGSPICE or not Path(NGSPICE).is_file():
    raise FileNotFoundError("Set NGSPICE to your ngspice executable.")

print("Using:", NGSPICE)

Using: C:\Users\patri\Downloads\ngspice-47_64\Spice64\bin\ngspice.exe


In [12]:
def make_ngspice_reference(comps, node_index, cur_index):
    prefixes = {
        "R": "R", "C": "C", "L": "L",
        "V": "V", "I": "I", "ISC": "V",
        "VCVS": "E", "VCCS": "G",
        "CCVS": "H", "CCCS": "F",
        "D": "B",
    }

    # Generated names avoid conflicts with custom component names.
    spice_names = {
        comp[0]: f"{prefixes[get_type(comp[0])]}REF{k}"
        for k, comp in enumerate(comps)
    }

    spice_nodes = {
        node: ("0" if node == "0" else f"NREF{k}")
        for k, node in enumerate(node_index)
    }

    lines = ["Reference DC circuit"]

    for comp in comps:
        name = comp[0]
        kind = get_type(name)
        spice_name = spice_names[name]
        plus, minus = (spice_nodes[n] for n in comp[1:3])

        if kind == "ISC":
            lines.append(f"{spice_name} {plus} {minus} DC 0")

        elif kind in ("V", "I"):
            lines.append(
                f"{spice_name} {plus} {minus} DC {comp[-1]}"
            )

        elif kind in ("R", "C", "L"):
            lines.append(
                f"{spice_name} {plus} {minus} {comp[3]}"
            )

        elif kind in ("VCVS", "VCCS"):
            cp, cm = (spice_nodes[n] for n in comp[3:5])
            lines.append(
                f"{spice_name} {plus} {minus} {cp} {cm} {comp[5]}"
            )

        elif kind in ("CCVS", "CCCS"):
            sense = spice_names[comp[3]]
            lines.append(
                f"{spice_name} {plus} {minus} {sense} {comp[4]}"
            )
        elif kind == "D":
            lines.append(
                f"{spice_name} {plus} {minus} "
                f"I=1e-15*(exp(V({plus},{minus})/0.025)-1)"
            )

    expressions = {
        node: f"v({spice_nodes[node].lower()})"
        for node, index in node_index.items()
        if index is not None
    }

    for comp in comps:
        label = "I" + comp[0]

        if label in cur_index:
            spice_name = spice_names[comp[0]].lower()

            if get_type(comp[0]) == "L":
                expressions[label] = f"@{spice_name}[i]"
            else:
                expressions[label] = f"{spice_name}#branch"

    # Simple aliases make the reference output easier to parse.
    measurements = []

    lines += [".control", "set numdgt=17", "op"]

    for k, (label, expression) in enumerate(expressions.items()):
        alias = f"check_{k}"
        lines += [
            f"let {alias} = {expression}",
            f"print {alias}",
        ]
        measurements.append((label, alias))

    lines += ["quit", ".endc", ".end"]

    return "\n".join(lines) + "\n", measurements

In [13]:
def compare_case(path, atol=1e-8, rtol=1e-5):
    _, comps, _ = read_netlist(path)

    # Suppress the solver's usual voltage/current printout.
    with contextlib.redirect_stdout(io.StringIO()):
        G, C, b, nodes, currents, x = run_netlist(
            path, show_matrices=False
        )

    if x is None:
        raise ValueError(
            "Your solver reports a singular circuit; "
            "no unique solution to compare."
        )

    own = {
        name: float(x[index])
        for name, index in nodes.items()
        if index is not None
    }
    own.update({
        name: float(x[index])
        for name, index in currents.items()
    })

    reference_netlist, measurements = make_ngspice_reference(
        comps, nodes, currents
    )

    reference = run_ngspice(
        reference_netlist, NGSPICE, measurements
    )

    passed = True

    for name, actual in own.items():
        expected = reference[name]
        ok = (
            math.isfinite(actual)
            and math.isclose(
                actual, expected,
                abs_tol=atol, rel_tol=rtol
            )
        )
        passed = passed and ok

        print(
            f"{name:18} "
            f"MNA={actual: .9g}  "
            f"ngspice={expected: .9g}  "
            f"{'PASS' if ok else 'FAIL'}"
        )

    return passed


cases = sorted(Path("test_cases").glob("*.cir"))
results = []

if not cases:
    raise FileNotFoundError("No .cir files found in test_case.")

import traceback

for case in cases:
    print(f"\n--- {case.name} ---")

    try:
        results.append(compare_case(case))
    except Exception:
        results.append(False)
        traceback.print_exc()

print(f"\n{sum(results)}/{len(results)} test cases passed.")


--- 01_voltage_divider.cir ---
N1                 MNA= 10  ngspice= 10  PASS
N2                 MNA= 5  ngspice= 5  PASS
IVIN               MNA=-0.005  ngspice=-0.005  PASS

--- 02_current_source.cir ---
N1                 MNA=-2  ngspice=-2  PASS

--- 03_capacitor_dc.cir ---
N1                 MNA= 12  ngspice= 12  PASS
N2                 MNA= 8  ngspice= 8  PASS
IVIN               MNA=-0.004  ngspice=-0.004  PASS

--- 04_inductor_dc.cir ---
N1                 MNA= 9  ngspice= 9  PASS
N2                 MNA= 6  ngspice= 6  PASS
N3                 MNA= 6  ngspice= 6  PASS
IVIN               MNA=-0.003  ngspice=-0.003  PASS
IL1                MNA= 0.003  ngspice= 0.003  PASS

--- 05_voltage_source_polarity.cir ---
N1                 MNA=-5  ngspice=-5  PASS
N2                 MNA=-2.5  ngspice=-2.5  PASS
IVIN               MNA= 0.0025  ngspice= 0.0025  PASS

--- 06_vccs.cir ---
N1                 MNA= 4  ngspice= 4  PASS
N2                 MNA= 2  ngspice= 2  PASS
N3                 MN